# Human PlayerBot: Engine Selection via Logistic Regression

## Overview

The Human PlayerBot emulates human chess play at a target Elo by reproducing the **distribution of centipawn loss (CPL)** observed in real games.

Instead of directly predicting moves, the system uses a **mixture-of-experts architecture**:

* **Experts (engines):**

  * Strength-limited Stockfish
  * Maia (Elo-matched)
  * BlunderMaia (lower-Elo Maia)

* **Policy (logistic regression):**

  * Outputs a probability distribution over engines
  * Conditioned on:

    * Player Elo
    * Position complexity

**Inference pipeline:**

1. Sample an engine from the policy
2. Query that engine for a move

---

## Core Principle

Human play is better characterized by **error magnitude** than exact move choice.

We model:

P(move | x) = Σₑ P(e | x) · P(move | e)

Where:

* e = engine (expert)
* x = (Elo, position complexity)

The policy is trained so that the resulting move distribution matches **human centipawn loss behavior**.

---

## Engines (Experts)

### 1. Stockfish (Strength-Limited)

* Produces stable, tactically sound moves
* Low centipawn loss
* Acts as a high-quality anchor

---

### 2. Maia (Elo-Matched)

* Trained on human games
* Captures human positional preferences
* Used **without temperature** (preserves calibration)
* Represents baseline human play

---

### 3. BlunderMaia (Lower-Elo Maia)

* Typically ~300 Elo below target
* Produces structured mistakes
* Captures:

  * inaccuracies
  * mistakes
  * blunders

This replaces artificial temperature with **empirical human error distributions**.

---

## Feature Space

Each position is encoded using:

* **Elo** (target player rating)
* **Position complexity** (precomputed function)

Complexity captures difficulty signals such as:

* evaluation spread across candidate moves
* tactical volatility
* number of viable alternatives

---

## Label Construction

For each training position:

1. Let m_h be the human move

2. Compute human centipawn loss:

   Δ_h = CPL(m_h)

3. For each engine e:

   * Generate move m_e
   * Compute:

     Δ_e = CPL(m_e)

4. Assign label:

   y = argmin_e |Δ_e − Δ_h|

This selects the engine whose move best matches the **human error magnitude**.

---

## Model

We train a **multinomial logistic regression** model.

The model learns:

P(e | x) = softmax(Wx)

Where:

* x = (Elo, complexity)
* e ∈ {Stockfish, Maia, BlunderMaia}

---

## Training Objective

* Standard cross-entropy loss on engine labels
* Implicit goal:

  * Match the **conditional distribution of centipawn loss** given x

---

## Inference Pipeline

Given a position:

1. Compute features:

   * Elo
   * Complexity

2. Evaluate policy:

   π = P(e | x)

3. Sample engine:

```python
engine = np.random.choice(["sf", "maia", "blunder"], p=pi)
```

4. Query selected engine → return move

---

## Design Rationale

### 1. Centipawn Loss as Ground Truth

* Move matching is brittle and ambiguous
* CPL captures **decision quality**
* Human skill strongly correlates with CPL distribution

---

### 2. Mixture of Experts

* Each engine represents a **region of the error distribution**:

  * Stockfish → low error
  * Maia → moderate, human-like error
  * BlunderMaia → high error, heavy tail

* Logistic regression learns how to combine them

---

### 3. No Temperature on Maia

* Maia already encodes a calibrated human policy
* Temperature distorts its distribution
* Variability is introduced via **multiple Maia Elo levels**, not softmax scaling

---

### 4. Structured Errors (Not Random Noise)

* Human mistakes are systematic, not random
* Lower-Elo Maia preserves:

  * positional misunderstandings
  * tactical oversights

---

### 5. Conditioning on Complexity

* Humans blunder more in difficult positions
* Complexity shifts probability mass toward weaker experts

---

## Resulting Behavior

The system reproduces:

* Correct **average playing strength**
* Realistic **error frequency**
* Heavy-tailed **blunder distribution**
* Human-like **move selection patterns**

---

## Summary

The Human PlayerBot models chess play as:

> A probabilistic mixture of engines, trained to match human centipawn loss distributions conditioned on Elo and position complexity.

It achieves realism by:

* using **human-trained models (Maia)**
* avoiding artificial noise
* learning **how to combine policies**, rather than replacing them
